# Graph-Based Differential Privacy Pipeline
### Privacy-Preserving Machine Learning via Graph Edge Perturbation for IIoT Data
**Author:** Mahdi Mohammad Shibli · **Year:** 2026

---

## Pipeline

```
LOCAL (trusted — data owner)                         CLOUD (untrusted)
──────────────────────────────────────────────────   ─────────────────────
1. Load full dataset (train + test)        PRIVATE
2. Build KNN graph W on ALL data           PRIVATE
3. Perturb graph edges with ε-DP → W_pert  PRIVATE   ← NEW: privacy here
4. Smooth features using W_pert → X_smooth PRIVATE
5. PCA compress → Z_smooth                 PRIVATE
6. Split at n_train                        PRIVATE
   ──────────────────────────────────────────────→   Z_train_smooth (upload)
                                                 →   Z_test_smooth  (upload)
                                                      Classifier training + eval
```

## Key Difference from Feature-Space DP

| Aspect | Feature-Space DP (old) | Graph-Edge DP (new) |
|--------|------------------------|---------------------|
| Where noise is added | PCA components | Graph edge weights |
| What is protected | Individual feature values | Pairwise similarity structure |
| Privacy guarantee | Per-sample feature values | Per-edge (who is similar to whom) |
| Mechanism | Laplace/Gaussian on Z | Randomised response on W |
| Formal guarantee | ε-DP on features | ε-DP on edges |

## Cell Map

| Cell | Purpose |
|------|---------|
| 0 | Install |
| 1 | Imports |
| 2 | Configuration |
| 3 | Data + group-stratified split |
| 4 | Build clean KNN graph W_clean |
| 4b | Perturb graph edges → W_pert (NEW — privacy here) |
| 4c | Smooth features using W_pert → X_smooth → PCA → Z_smooth |
| 5 | SNR & graph perturbation diagnostics |
| 6 | Classifier suite |
| 7 | Attribute inference attack |
| 8 | Full epsilon sweep |
| 9 | Visualisation |

## Cell 0 — Install

In [11]:
# !pip install networkx --quiet
# print("✓ Dependencies ready")

## Cell 1 — Imports

In [12]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

from scipy.sparse.csgraph     import laplacian as sparse_laplacian
from sklearn.decomposition    import PCA
from sklearn.neighbors        import kneighbors_graph, KNeighborsClassifier
from sklearn.preprocessing    import StandardScaler, LabelEncoder
from sklearn.model_selection  import StratifiedShuffleSplit
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm              import SVC
from sklearn.metrics          import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✓ All imports successful")

✓ All imports successful


## Cell 2 — Configuration
> **Edit this cell only.**

In [13]:
CFG = {
    # ── Data ──────────────────────────────────────────────────────────────
    "data_path"               : "features_raw.csv",
    "label_column"            : "fault_type",
    "group_column"            : "file_id",
    "sensitive_class"         : 0,               # 0 = Ball fault
    "class_names"             : ["B", "IR", "Normal", "OR"],
    "test_size"               : 0.2,
    "random_state"            : 42,

    # ── Graph ─────────────────────────────────────────────────────────────
    "k_neighbors"             : 7,              # KNN connectivity
    "alpha"                   : 0.7,             # smoothing strength

    # ── Graph Privacy (NEW) ───────────────────────────────────────────────
    # Privacy is now applied at the GRAPH level (edge perturbation)
    # rather than the feature level.
    #
    # Edge-level ε-DP (randomised response on edges):
    #   Existing edge: keep with prob p_keep = e^ε / (1 + e^ε)
    #   Non-edge:      add  with prob p_add  = 1   / (1 + e^ε)
    #
    # Smaller ε → more edge flipping → stronger privacy, weaker smoothing
    # Larger  ε → fewer flips → weaker privacy, better smoothing quality
    # "graph_epsilons"          : [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0, 100.0],
    "graph_epsilons"          : [0.1,  1.0, 2.0, 5.0,],

    # ── PCA (applied AFTER graph smoothing) ───────────────────────────────
    "pca_variance"            : 0.90,
    "pca_max_components"      : 50,

    # ── Evaluation ────────────────────────────────────────────────────────
    "plot_cms_sweep"          : False,
    "epsilon_highlight"       : 1.0,
}

print("✓ Configuration loaded")
print(f"  Sensitive class     : {CFG['sensitive_class']} ({CFG['class_names'][CFG['sensitive_class']]})")
print(f"  Graph epsilon range : {CFG['graph_epsilons']}")
print(f"  PCA variance target : {CFG['pca_variance']*100:.0f}%")
print(f"  Privacy mechanism   : Edge-level ε-DP (randomised response)")

✓ Configuration loaded
  Sensitive class     : 0 (B)
  Graph epsilon range : [0.1, 1.0, 2.0, 5.0]
  PCA variance target : 90%
  Privacy mechanism   : Edge-level ε-DP (randomised response)


## Cell 3 — Data Loading + Group-Stratified Split
Split at **group level** (recording file) to prevent window overlap leakage.

In [14]:
df     = pd.read_csv(CFG["data_path"])
y      = LabelEncoder().fit_transform(df[CFG["label_column"]])
X      = df.drop(columns=[CFG["label_column"], CFG["group_column"]]).values
groups = df[CFG["group_column"]].values

group_ids    = np.unique(groups)
group_labels = np.array([int(pd.Series(y[groups == g]).mode()[0]) for g in group_ids])

print("Group-level class distribution:")
for cls, name in enumerate(CFG["class_names"]):
    print(f"  Class {cls} ({name:<8}) : {int((group_labels==cls).sum())} groups")

for cls in np.unique(group_labels):
    if int((group_labels==cls).sum()) < 2:
        raise ValueError(f"Class {CFG['class_names'][cls]} has < 2 groups.")

sss = StratifiedShuffleSplit(n_splits=1, test_size=CFG["test_size"],
                              random_state=CFG["random_state"])
tr_gi, te_gi = next(sss.split(group_ids, group_labels))
train_groups = set(group_ids[tr_gi])
test_groups  = set(group_ids[te_gi])
assert len(train_groups & test_groups) == 0, "Leakage!"

train_mask = np.isin(groups, list(train_groups))
test_mask  = np.isin(groups, list(test_groups))

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

X_all_sc = np.vstack([X_train_sc, X_test_sc])
y_all    = np.concatenate([y_train, y_test])
n_train  = len(X_train_sc)

print(f"\n✓ Split complete — no group overlap")
print(f"  Train : {X_train_sc.shape}  |  Test : {X_test_sc.shape}  |  All : {X_all_sc.shape}")
for split_name, y_split in [("Train", y_train), ("Test", y_test)]:
    pcts = " | ".join(f"{n}:{(y_split==c).mean()*100:.1f}%" for c,n in enumerate(CFG["class_names"]))
    print(f"  {split_name:<8} {pcts}")

Group-level class distribution:
  Class 0 (B       ) : 16 groups
  Class 1 (IR      ) : 16 groups
  Class 2 (Normal  ) : 4 groups
  Class 3 (OR      ) : 28 groups

✓ Split complete — no group overlap
  Train : (13721, 129)  |  Test : (3782, 129)  |  All : (17503, 129)
  Train    B:22.4% | IR:22.4% | Normal:17.2% | OR:38.0%
  Test     B:18.7% | IR:18.7% | Normal:25.0% | OR:37.6%


## Cell 4 — Build Clean KNN Graph
Builds the RBF-weighted KNN graph on all N samples.
This clean graph `W_clean` is used as the starting point for edge perturbation.
It never leaves the device in any form.

In [15]:
def build_sparse_knn_laplacian(X_scaled, k):
    """Build sparse RBF-weighted KNN graph + Laplacian. Returns (W, L, sigma)."""
    knn   = kneighbors_graph(X_scaled, n_neighbors=k,
                              mode='distance', include_self=False, n_jobs=-1)
    dist  = knn.data
    sigma = float(np.median(dist)) or 1.0
    knn.data = np.exp(-(dist**2) / (2 * sigma**2))
    W = knn.maximum(knn.T).tocsr()
    L = sparse_laplacian(W, normed=False).tocsr()
    return W, L, sigma


def smooth_features_sparse(X, L_csr, alpha=0.5):
    """X_smooth = X - α · D⁻¹ · L · X  (sparse — no dense N×N)."""
    deg   = np.array(L_csr.diagonal(), dtype=np.float64)
    deg   = np.where(deg == 0, 1e-10, deg)
    D_inv = sp.diags(1.0 / deg)
    return X - alpha * D_inv.dot(L_csr).dot(X)


N_all = X_all_sc.shape[0]
print(f"Building clean KNN graph (N={N_all:,}, k={CFG['k_neighbors']})...")
W_clean, L_clean, sigma = build_sparse_knn_laplacian(X_all_sc, k=CFG["k_neighbors"])

dense_mb  = (N_all * N_all * 8) / 1e6
sparse_mb = (W_clean.nnz * 8 * 3) / 1e6
diff      = W_clean - W_clean.T
is_sym    = np.allclose(diff.data, 0, atol=1e-8) if diff.nnz > 0 else True
row_sums  = np.array(L_clean.sum(axis=1)).ravel()

print(f"  Non-zeros : {W_clean.nnz:,}  |  {dense_mb:.0f} MB → {sparse_mb:.1f} MB  ({dense_mb/sparse_mb:.0f}× reduction)")
print(f"  Symmetric : {is_sym}  |  Zero row-sum : {np.allclose(row_sums,0,atol=1e-6)}")
print(f"  RBF sigma : {sigma:.6f}")

# ── Baseline: clean smoothing (no privacy, used for comparison) ───────────
print("\nComputing clean smoothing baseline...")
X_all_smooth_clean   = smooth_features_sparse(X_all_sc, L_clean, CFG["alpha"])
X_train_smooth_clean = X_all_smooth_clean[:n_train]
X_test_smooth_clean  = X_all_smooth_clean[n_train:]
print(f"  X_all_smooth_clean : {X_all_smooth_clean.shape}")
print("\n✓ Cell 4 complete — W_clean and X_all_smooth_clean ready")

Building clean KNN graph (N=17,503, k=7)...
  Non-zeros : 185,088  |  2451 MB → 4.4 MB  (552× reduction)
  Symmetric : True  |  Zero row-sum : True
  RBF sigma : 3.851238

Computing clean smoothing baseline...
  X_all_smooth_clean : (17503, 129)

✓ Cell 4 complete — W_clean and X_all_smooth_clean ready


## Cell 4b — Graph Edge Perturbation (ε-DP) ← CORE PRIVACY CELL

This is where privacy is enforced. Each edge in the clean graph is independently
randomised using the **randomised response** mechanism:

```
Existing edge (w_ij > 0): keep with prob  p_keep = e^ε / (1 + e^ε)
Non-edge     (w_ij = 0):  add  with prob  p_add  = 1   / (1 + e^ε)
```

**Formal guarantee:** This satisfies **edge-level ε-DP** (Warner 1965;
Hay et al. 2009). An adversary who observes the perturbed graph W_pert
cannot determine with high confidence whether any specific edge exists
in the original graph — and therefore cannot determine whether any two
samples are truly similar (e.g., both Ball faults).

**Why edge-level DP protects Ball fault identity:**
If two Ball fault samples were connected in W_clean (because they are
similar), that edge may be removed in W_pert. Their smoothed features
will no longer blend with each other — making Ball fault samples look
less like a coherent cluster and more like random samples.

In [16]:
def perturb_graph_edges(W_sparse, epsilon):
    """
    Apply edge-level ε-DP via randomised response on the adjacency matrix.

    For each existing edge (i,j) with weight w_ij:
        Keep with probability  p_keep = e^ε / (1 + e^ε)
        Remove otherwise       (probability 1 - p_keep)

    For each non-edge (i,j) where w_ij = 0:
        Add (mean weight) with probability  p_add = 1 / (1 + e^ε)
        Keep absent otherwise

    This satisfies ε-DP at the edge level via randomised response.
    Reference: Warner (1965); Hay et al. (2009) Accurate Estimation
    of Short-Range Statistics in Network Graphs Under DP.

    Args:
        W_sparse : Clean symmetric adjacency matrix (CSR sparse)
        epsilon  : Privacy budget for edge perturbation

    Returns:
        W_pert   : Perturbed symmetric adjacency matrix (CSR sparse)
        stats    : dict with perturbation statistics
    """
    eps_clip = float(np.clip(epsilon, 1e-6, 500))
    p_keep   = np.exp(eps_clip) / (1.0 + np.exp(eps_clip))   # prob keep existing edge
    p_add    = 1.0 / (1.0 + np.exp(eps_clip))                # prob add non-edge

    W_coo    = W_sparse.tocoo()
    N        = W_sparse.shape[0]
    mean_w   = float(W_coo.data.mean()) if len(W_coo.data) > 0 else 1.0

    # ── Step 1: Process existing edges — keep or remove ──────────────────
    keep_mask   = np.random.random(len(W_coo.data)) < p_keep
    kept_rows   = W_coo.row[keep_mask]
    kept_cols   = W_coo.col[keep_mask]
    kept_data   = W_coo.data[keep_mask]

    n_removed   = int((~keep_mask).sum())

    # ── Step 2: Add new edges (non-edges → edges with prob p_add) ────────
    # Sample candidate non-edges — same order of magnitude as existing edges
    # to keep the graph roughly the same density
    n_sample    = W_sparse.nnz      # sample as many candidates as existing edges
    cand_i      = np.random.randint(0, N, size=n_sample)
    cand_j      = np.random.randint(0, N, size=n_sample)

    # Filter: must be off-diagonal and not already an existing edge
    existing_set = set(zip(W_coo.row.tolist(), W_coo.col.tolist()))
    add_mask     = np.array([
        i != j and (i, j) not in existing_set
        for i, j in zip(cand_i, cand_j)
    ])
    cand_i = cand_i[add_mask]
    cand_j = cand_j[add_mask]

    # Apply randomised response — add each candidate with prob p_add
    add_keep  = np.random.random(len(cand_i)) < p_add
    add_i     = cand_i[add_keep]
    add_j     = cand_j[add_keep]
    add_data  = np.full(len(add_i), mean_w)    # assign mean weight to new edges

    n_added   = len(add_i)

    # ── Step 3: Assemble perturbed graph ──────────────────────────────────
    all_rows = np.concatenate([kept_rows, add_i])
    all_cols = np.concatenate([kept_cols, add_j])
    all_data = np.concatenate([kept_data, add_data])

    W_pert   = sp.csr_matrix((all_data, (all_rows, all_cols)), shape=(N, N))
    W_pert   = W_pert.maximum(W_pert.T)    # enforce symmetry

    flip_rate = (n_removed + n_added) / max(W_sparse.nnz, 1)

    return W_pert, {
        "epsilon"    : epsilon,
        "p_keep"     : round(p_keep, 4),
        "p_add"      : round(p_add, 4),
        "n_original" : W_sparse.nnz,
        "n_removed"  : n_removed,
        "n_added"    : n_added,
        "n_final"    : W_pert.nnz,
        "flip_rate"  : round(flip_rate, 4),
    }


def smooth_and_pca(X_all_sc, W_pert, alpha, pca_obj=None,
                    fit_pca=False, n_train=None):
    """
    Apply graph smoothing with perturbed graph, then PCA compress.

    Args:
        X_all_sc  : Scaled raw features (N × d)
        W_pert    : Perturbed adjacency matrix
        alpha     : Smoothing strength
        pca_obj   : Pre-fitted PCA object (None if fit_pca=True)
        fit_pca   : If True, fit PCA on training portion
        n_train   : Training set size (required if fit_pca=True)

    Returns:
        Z_all     : PCA-compressed smoothed features (N × k)
        pca_obj   : Fitted PCA object
    """
    L_pert    = sparse_laplacian(W_pert, normed=False).tocsr()
    X_smooth  = smooth_features_sparse(X_all_sc, L_pert, alpha)

    if fit_pca:
        pca_obj = PCA(n_components=pca_obj, random_state=42)
        pca_obj.fit(X_smooth[:n_train])

    Z_all = pca_obj.transform(X_smooth)
    return Z_all, pca_obj


# ── Determine PCA components from clean baseline ──────────────────────────
print("Determining optimal PCA components from clean baseline...")
pca_probe = PCA(n_components=min(CFG["pca_max_components"],
                                  X_all_smooth_clean.shape[1]))
pca_probe.fit(X_train_smooth_clean)
cum_var = np.cumsum(pca_probe.explained_variance_ratio_)
k_opt   = int(np.searchsorted(cum_var, CFG["pca_variance"]) + 1)
k_opt   = min(k_opt, CFG["pca_max_components"])
print(f"  PCA: {X_all_smooth_clean.shape[1]} → {k_opt} components ({cum_var[k_opt-1]*100:.1f}% variance)")

# ── Fit reference PCA on clean smoothed training data ─────────────────────
pca_ref = PCA(n_components=k_opt, random_state=CFG["random_state"])
pca_ref.fit(X_train_smooth_clean)

# ── Compute clean baseline Z (no privacy) ─────────────────────────────────
Z_clean_all   = pca_ref.transform(X_all_smooth_clean)
Z_clean_train = Z_clean_all[:n_train]
Z_clean_test  = Z_clean_all[n_train:]
print(f"  Z_clean shape : {Z_clean_all.shape}")

# ── Preview perturbation at ε=1.0 ─────────────────────────────────────────
print(f"\nPreview: graph perturbation at ε=1.0")
W_preview, stats = perturb_graph_edges(W_clean, epsilon=1.0)
print(f"  p_keep    : {stats['p_keep']}  (prob existing edge survives)")
print(f"  p_add     : {stats['p_add']}   (prob new edge is added)")
print(f"  Edges removed : {stats['n_removed']:,} / {stats['n_original']:,}  ({stats['n_removed']/stats['n_original']*100:.1f}%)")
print(f"  Edges added   : {stats['n_added']:,}")
print(f"  Flip rate     : {stats['flip_rate']:.4f}")
print(f"\n✓ Cell 4b complete — ready for epsilon sweep in Cell 8")

Determining optimal PCA components from clean baseline...
  PCA: 129 → 31 components (90.4% variance)
  Z_clean shape : (17503, 31)

Preview: graph perturbation at ε=1.0
  p_keep    : 0.7311  (prob existing edge survives)
  p_add     : 0.2689   (prob new edge is added)
  Edges removed : 49,781 / 185,088  (26.9%)
  Edges added   : 49,450
  Flip rate     : 0.5361

✓ Cell 4b complete — ready for epsilon sweep in Cell 8


## Cell 5 — Graph Perturbation Diagnostics
Shows how edge flip rate and graph structure change with epsilon.

In [17]:
print("=" * 68)
print("GRAPH PERTURBATION DIAGNOSTICS — Edge-Level ε-DP")
print("=" * 68)
print(f"  Clean graph: {W_clean.nnz:,} edges | N={N_all:,} nodes")
print(f"  k={CFG['k_neighbors']}, RBF sigma={sigma:.4f}")
print()
print(f"  {'ε':<8} {'p_keep':>8} {'p_add':>8} {'Removed':>10} {'Added':>10} {'Flip%':>8} {'Final edges':>12}")
print(f"  {'─'*8} {'─'*8} {'─'*8} {'─'*10} {'─'*10} {'─'*8} {'─'*12}")

for eps in CFG["graph_epsilons"]:
    eps_clip = float(np.clip(eps, 1e-6, 500))
    p_keep   = np.exp(eps_clip) / (1.0 + np.exp(eps_clip))
    p_add    = 1.0 / (1.0 + np.exp(eps_clip))
    # Expected values (no need to actually perturb)
    exp_removed = W_clean.nnz * (1 - p_keep)
    exp_added   = W_clean.nnz * p_add         # approx (sampling same number)
    exp_final   = W_clean.nnz - exp_removed + exp_added
    flip_rate   = (exp_removed + exp_added) / W_clean.nnz
    print(f"  {eps:<8} {p_keep:>8.4f} {p_add:>8.4f} "
          f"{exp_removed:>10.0f} {exp_added:>10.0f} "
          f"{flip_rate*100:>7.1f}% {exp_final:>12.0f}")

print(f"\n  Interpretation:")
print(f"  ε=0.01 → p_keep=0.50 → ~50% of edges flipped  (maximum privacy)")
print(f"  ε=1.0  → p_keep=0.73 → ~27% of edges flipped  (moderate privacy)")
print(f"  ε=5.0  → p_keep=0.99 → ~1%  of edges flipped  (near original graph)")
print(f"  ε=100  → p_keep≈1.00 → almost no flipping      (no privacy)")

GRAPH PERTURBATION DIAGNOSTICS — Edge-Level ε-DP
  Clean graph: 185,088 edges | N=17,503 nodes
  k=7, RBF sigma=3.8512

  ε          p_keep    p_add    Removed      Added    Flip%  Final edges
  ──────── ──────── ──────── ────────── ────────── ──────── ────────────
  0.1        0.5250   0.4750      87921      87921    95.0%       185088
  1.0        0.7311   0.2689      49778      49778    53.8%       185088
  2.0        0.8808   0.1192      22063      22063    23.8%       185088
  5.0        0.9933   0.0067       1239       1239     1.3%       185088

  Interpretation:
  ε=0.01 → p_keep=0.50 → ~50% of edges flipped  (maximum privacy)
  ε=1.0  → p_keep=0.73 → ~27% of edges flipped  (moderate privacy)
  ε=5.0  → p_keep=0.99 → ~1%  of edges flipped  (near original graph)
  ε=100  → p_keep≈1.00 → almost no flipping      (no privacy)


## Cell 6 — Classifier Suite

In [18]:
try:
    from cuml.ensemble     import RandomForestClassifier  as cuRF
    from cuml.ensemble     import GradientBoostingClassifier as cuGB
    from cuml.svm          import SVC                     as cuSVC
    from cuml.neighbors    import KNeighborsClassifier    as cuKNN
    from cuml.linear_model import LogisticRegression      as cuLR
    USE_GPU = True
    print("✓ cuML available — using GPU classifiers")
except ImportError:
    USE_GPU = False
    print("⚠  cuML not available — using CPU classifiers")

if USE_GPU:
    CLASSIFIERS = {
        "Logistic Regression" : cuLR(max_iter=500),
        "Random Forest"       : cuRF(n_estimators=100, random_state=CFG["random_state"]),
        "SVM"                 : cuSVC(kernel="rbf", probability=True),
        "KNN"                 : cuKNN(n_neighbors=5),
        "Gradient Boosting"   : cuGB(n_estimators=100, random_state=CFG["random_state"]),
    }
else:
    CLASSIFIERS = {
        "Logistic Regression" : LogisticRegression(max_iter=500, solver="saga",
                                                    random_state=CFG["random_state"]),
        "Random Forest"       : RandomForestClassifier(n_estimators=100,
                                                        random_state=CFG["random_state"]),
        "SVM"                 : SVC(kernel="rbf", probability=True,
                                    random_state=CFG["random_state"]),
        "KNN"                 : KNeighborsClassifier(n_neighbors=5),
        "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=100,
                                                             random_state=CFG["random_state"]),
    }


def evaluate_classifier(clf, X_te, y_te):
    y_pred = clf.predict(X_te)
    if hasattr(y_pred, "to_numpy"): y_pred = y_pred.to_numpy()
    n_cls  = len(np.unique(y_te))
    try:
        proba = clf.predict_proba(X_te)
        if hasattr(proba, "to_numpy"): proba = proba.to_numpy()
        auc = (roc_auc_score(y_te, proba[:,1]) if n_cls==2
               else roc_auc_score(y_te, proba, multi_class="ovr", average="macro"))
    except Exception: auc = np.nan
    return {
        "accuracy"    : round(accuracy_score(y_te, y_pred), 4),
        "f1_macro"    : round(f1_score(y_te, y_pred, average="macro",    zero_division=0), 4),
        "f1_weighted" : round(f1_score(y_te, y_pred, average="weighted", zero_division=0), 4),
        "precision"   : round(precision_score(y_te, y_pred, average="weighted", zero_division=0), 4),
        "recall"      : round(recall_score(y_te, y_pred, average="weighted",    zero_division=0), 4),
        "roc_auc"     : round(float(auc), 4) if not np.isnan(auc) else np.nan,
        "conf_matrix" : confusion_matrix(y_te, y_pred),
    }


def run_classifiers(X_tr, X_te, y_tr, y_te, label="", class_names=None, verbose=True):
    sc      = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)
    results = {}
    if verbose and label:
        print(f"\n  {'Classifier':<25} {'Acc':>6} {'F1-Mac':>7} {'F1-Wt':>7} {'Prec':>6} {'Rec':>6} {'AUC':>7}")
        print(f"  {'─'*25} {'─'*6} {'─'*7} {'─'*7} {'─'*6} {'─'*6} {'─'*7}")
    for name, clf in CLASSIFIERS.items():
        clf.fit(X_tr_sc, y_tr)
        m = evaluate_classifier(clf, X_te_sc, y_te)
        results[name] = m
        if verbose:
            auc_s = f"{m['roc_auc']:>7.4f}" if not np.isnan(m["roc_auc"]) else "   N/A "
            print(f"  {name:<25} {m['accuracy']:>6.4f} {m['f1_macro']:>7.4f} "
                  f"{m['f1_weighted']:>7.4f} {m['precision']:>6.4f} {m['recall']:>6.4f} {auc_s}")
    return results


def results_to_df(results):
    rows = [{"classifier": k, **{m: v for m, v in met.items() if m != "conf_matrix"}}
            for k, met in results.items()]
    df = pd.DataFrame(rows).set_index("classifier")
    return df[["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc"]]


print("=" * 70)
print("BASELINE — clean graph smoothing + PCA (no edge perturbation)")
print("=" * 70)
baseline_results = run_classifiers(Z_clean_train, Z_clean_test, y_train, y_test,
                                    label="Baseline", class_names=CFG["class_names"])
baseline_df = results_to_df(baseline_results)
print(f"\n  Best by F1-weighted : {baseline_df['f1_weighted'].idxmax()}  "
      f"({baseline_df['f1_weighted'].max():.4f})")

⚠  cuML not available — using CPU classifiers
BASELINE — clean graph smoothing + PCA (no edge perturbation)

  Classifier                   Acc  F1-Mac   F1-Wt   Prec    Rec     AUC
  ───────────────────────── ────── ─────── ─────── ────── ────── ───────
  Logistic Regression       0.8173  0.8138  0.8134 0.8170 0.8173  0.9243
  Random Forest             0.9268  0.9175  0.9228 0.9330 0.9268  0.9961
  SVM                       0.9601  0.9570  0.9590 0.9620 0.9601  0.9973
  KNN                       0.8612  0.8630  0.8676 0.9070 0.8612  0.9526
  Gradient Boosting         0.9273  0.9150  0.9234 0.9329 0.9273  0.9954

  Best by F1-weighted : SVM  (0.9590)


## Cell 7 — Attribute Inference Attack
Adversary uses worst-case (strongest) model across LR, RF, SVM.

In [21]:
def run_inference_attack(X_tr, X_te, y_tr, y_te, sensitive_class,
                         return_all=False):
    """
    Attribute Inference Attack — worst-case adversary.
    Trains LR, RF, SVM adversaries; reports highest accuracy.
    """
    y_s_tr = (y_tr == sensitive_class).astype(int)
    y_s_te = (y_te == sensitive_class).astype(int)

    sc      = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr)
    X_te_sc = sc.transform(X_te)

    adversaries = {
        "LR"  : LogisticRegression(max_iter=500, solver="saga",
                                    class_weight="balanced", random_state=42),
        # "RF"  : RandomForestClassifier(n_estimators=100,
        #                                 class_weight="balanced", random_state=42),
        # "SVM" : SVC(kernel="rbf", probability=False,
        #             class_weight="balanced", random_state=42),
    }

    rand_bl      = float(max(y_s_te.mean(), 1.0 - y_s_te.mean()))
    results_each = {}
    worst_acc    = 0.0
    worst_name   = "LR"

    for name, clf in adversaries.items():
        clf.fit(X_tr_sc, y_s_tr)
        acc = accuracy_score(y_s_te, clf.predict(X_te_sc))
        results_each[name] = round(acc, 4)
        if acc > worst_acc:
            worst_acc  = acc
            worst_name = name

    out = {
        "adversary_accuracy" : round(worst_acc, 4),
        "worst_adversary"    : worst_name,
        "random_baseline"    : round(rand_bl, 4),
        "privacy_gain"       : round(rand_bl - worst_acc, 4),
    }
    if return_all:
        out["per_adversary"] = results_each
    return out


print("Adversary sanity checks (before any graph perturbation):")
print(f"  {'Label':<40} {'Worst adv':>10} {'Model':>6} {'Baseline':>10} {'Gain':>8}  {'LR':>7} {'RF':>7} {'SVM':>7}")
print(f"  {'─'*40} {'─'*10} {'─'*6} {'─'*10} {'─'*8}  {'─'*7} {'─'*7} {'─'*7}")

for label, Xtr, Xte in [
    ("Raw scaled features",              X_train_sc,    X_test_sc),
    ("Clean graph-smooth + PCA",         Z_clean_train, Z_clean_test),
]:
    p  = run_inference_attack(Xtr, Xte, y_train, y_test,
                               CFG["sensitive_class"], return_all=True)
    ea = p["per_adversary"]
    s  = "✓ protecting" if p["privacy_gain"] > 0 else "✗ leaking"
    print(f"  {label:<40} {p['adversary_accuracy']:>10.4f} "
          f"{p['worst_adversary']:>6} "
          f"{p['random_baseline']:>10.4f} "
          f"{p['privacy_gain']:>+8.4f}  "
        #   f"{ea['LR']:>7.4f} {ea['RF']:>7.4f} {ea['SVM']:>7.4f}  {s}"
    )

Adversary sanity checks (before any graph perturbation):
  Label                                     Worst adv  Model   Baseline     Gain       LR      RF     SVM
  ──────────────────────────────────────── ────────── ────── ────────── ────────  ─────── ─────── ───────
  Raw scaled features                          0.9804     LR     0.8131  -0.1674  
  Clean graph-smooth + PCA                     0.9162     LR     0.8131  -0.1031  


## Cell 8 — Full Epsilon Sweep (Graph Edge Perturbation)

For each ε:
1. Perturb W_clean edges → W_pert  *(edge-level ε-DP, LOCAL)*
2. Smooth X_all_sc using W_pert → X_smooth_pert  *(LOCAL)*
3. PCA → Z_pert  *(LOCAL)*
4. Split → Z_train_pert | Z_test_pert  *(upload to cloud)*
5. Classify → utility  |  Adversary → privacy

In [22]:
import time

def run_graph_epsilon_sweep(
    X_all_sc, y_train, y_test, n_train,
    W_clean, pca_obj, alpha,
    sensitive_class=0, graph_epsilons=None,
    class_names=None, plot_cms_sweep=False,
    baseline_results=None,
):
    """
    Full epsilon sweep — privacy via graph edge perturbation.

    For each epsilon:
        1. Perturb W_clean → W_pert  (edge-level ε-DP)
        2. Smooth X_all_sc with W_pert → X_smooth
        3. PCA transform → Z_pert
        4. Classify + adversary attack
    """
    if graph_epsilons is None:
        graph_epsilons = CFG["graph_epsilons"]

    records       = []
    sweep_records = []

    # ── Baseline ──────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("  BASELINE — clean graph (no edge perturbation)")
    print("="*70)

    if baseline_results is None:
        Z_clean = pca_obj.transform(
            smooth_features_sparse(X_all_sc, sparse_laplacian(W_clean, normed=False).tocsr(), alpha)
        )
        base_clf = run_classifiers(Z_clean[:n_train], Z_clean[n_train:],
                                    y_train, y_test, label="Baseline",
                                    class_names=class_names)
        base_prv = run_inference_attack(Z_clean[:n_train], Z_clean[n_train:],
                                         y_train, y_test, sensitive_class)
    else:
        base_clf, base_prv = baseline_results
        print("  (Baseline reused from pre-computation)")

    base_df = results_to_df(base_clf)
    best_b  = base_df["f1_weighted"].idxmax()
    records.append({
        "epsilon": 0.0, "label": "Baseline (clean)", "best_classifier": best_b,
        **{k: base_df.loc[best_b, k]
           for k in ["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc"]},
        **base_prv,
        "p_keep": 1.0, "p_add": 0.0, "flip_rate": 0.0,
        "edges_removed": 0, "edges_added": 0,
    })

    # ── Epsilon loop ──────────────────────────────────────────────────────
    for i, eps in enumerate(graph_epsilons):
        t0 = time.time()
        print(f"\n{'─'*70}")
        print(f"  ε = {eps:<8}  |  graph edge perturbation  [{i+1}/{len(graph_epsilons)}]")
        print("─"*70)

        # LOCAL: perturb graph edges with ε-DP
        W_pert, gstats = perturb_graph_edges(W_clean, epsilon=eps)

        print(f"  p_keep={gstats['p_keep']:.4f}  p_add={gstats['p_add']:.4f}  "
              f"flip_rate={gstats['flip_rate']:.4f}  "
              f"removed={gstats['n_removed']:,}  added={gstats['n_added']:,}")

        # LOCAL: smooth with perturbed graph, then PCA
        L_pert   = sparse_laplacian(W_pert, normed=False).tocsr()
        X_smooth = smooth_features_sparse(X_all_sc, L_pert, alpha)
        Z_pert   = pca_obj.transform(X_smooth)    # use reference PCA

        Z_tr_pert = Z_pert[:n_train]
        Z_te_pert = Z_pert[n_train:]

        # CLOUD: classify
        clf_res = run_classifiers(Z_tr_pert, Z_te_pert, y_train, y_test,
                                   label=f"ε={eps}", class_names=class_names)
        clf_df  = results_to_df(clf_res)
        best_c  = clf_df["f1_weighted"].idxmax()

        # PRIVACY: adversary attack
        prv = run_inference_attack(Z_tr_pert, Z_te_pert,
                                    y_train, y_test, sensitive_class)

        elapsed = time.time() - t0
        print(f"\n  Best: {best_c:<25}  adv={prv['adversary_accuracy']:.4f}  "
              f"baseline={prv['random_baseline']:.4f}  "
              f"gain={prv['privacy_gain']:+.4f}  "
              f"({'✓' if prv['privacy_gain']>0 else '✗'})  [{elapsed:.1f}s]")

        if plot_cms_sweep:
            plot_confusion_matrices(clf_res, class_names, label=f"ε={eps}")

        row = {
            "epsilon"        : eps,
            "label"          : f"ε={eps}",
            "best_classifier": best_c,
            **{k: clf_df.loc[best_c, k]
               for k in ["accuracy","f1_macro","f1_weighted","precision","recall","roc_auc"]},
            **prv,
            "p_keep"         : gstats["p_keep"],
            "p_add"          : gstats["p_add"],
            "flip_rate"      : gstats["flip_rate"],
            "edges_removed"  : gstats["n_removed"],
            "edges_added"    : gstats["n_added"],
        }
        records.append(row)
        sweep_records.append(row)

    df = pd.DataFrame(records)
    print("\n\n" + "="*70)
    print("  RESULTS — Graph Edge Perturbation ε-DP")
    print("="*70)
    print(df[["label","best_classifier","accuracy","f1_weighted",
              "roc_auc","adversary_accuracy","privacy_gain","flip_rate"]]
          .to_string(index=False))
    return df, sweep_records


# ── Compute baseline once ─────────────────────────────────────────────────
print("Computing baseline (clean graph, no perturbation)...")
t0 = time.time()
_base_clf = run_classifiers(
    Z_clean_train, Z_clean_test, y_train, y_test,
    label="Baseline", class_names=CFG["class_names"],
)
_base_prv = run_inference_attack(
    Z_clean_train, Z_clean_test, y_train, y_test,
    CFG["sensitive_class"],
)
print(f"  Baseline done in {time.time()-t0:.1f}s")

# ── Run sweep ─────────────────────────────────────────────────────────────
sweep_df, sweep_records = run_graph_epsilon_sweep(
    X_all_sc        = X_all_sc,
    y_train         = y_train,
    y_test          = y_test,
    n_train         = n_train,
    W_clean         = W_clean,
    pca_obj         = pca_ref,
    alpha           = CFG["alpha"],
    sensitive_class = CFG["sensitive_class"],
    graph_epsilons  = CFG["graph_epsilons"],
    class_names     = CFG["class_names"],
    plot_cms_sweep  = CFG["plot_cms_sweep"],
    baseline_results= (_base_clf, _base_prv),
)

Computing baseline (clean graph, no perturbation)...

  Classifier                   Acc  F1-Mac   F1-Wt   Prec    Rec     AUC
  ───────────────────────── ────── ─────── ─────── ────── ────── ───────
  Logistic Regression       0.8173  0.8138  0.8134 0.8170 0.8173  0.9243
  Random Forest             0.9268  0.9175  0.9228 0.9330 0.9268  0.9961
  SVM                       0.9601  0.9570  0.9590 0.9620 0.9601  0.9973
  KNN                       0.8612  0.8630  0.8676 0.9070 0.8612  0.9526
  Gradient Boosting         0.9273  0.9150  0.9234 0.9329 0.9273  0.9954
  Baseline done in 57.5s

  BASELINE — clean graph (no edge perturbation)
  (Baseline reused from pre-computation)

──────────────────────────────────────────────────────────────────────
  ε = 0.1       |  graph edge perturbation  [1/4]
──────────────────────────────────────────────────────────────────────
  p_keep=0.5250  p_add=0.4750  flip_rate=0.9529  removed=88,333  added=88,040

  Classifier                   Acc  F1-Mac   F1-

## Cell 9 — Visualisation

In [ ]:
def plot_graph_privacy_tradeoff(df, title="Graph Edge ε-DP"):
    sweep = df[df["epsilon"] > 0].copy()
    base  = df[df["label"] == "Baseline (clean)"].iloc[0]

    fig, ax1 = plt.subplots(figsize=(12, 6))
    c1 = "tab:blue"
    ax1.set_xlabel("ε (log scale)\n← more private (more edge flips)    less private →", fontsize=12)
    ax1.set_ylabel("Utility Score", color=c1, fontsize=12)
    ax1.plot(sweep["epsilon"], sweep["f1_weighted"], color=c1, marker="o",
             linewidth=2.5, markersize=7, label="F1-Weighted")
    ax1.plot(sweep["epsilon"], sweep["accuracy"], color=c1, marker="s",
             linewidth=1.5, markersize=5, linestyle="--", alpha=0.6, label="Accuracy")
    ax1.axhline(base["f1_weighted"], color=c1, linestyle=":", alpha=0.4,
                linewidth=1.2, label=f"Baseline F1 ({base['f1_weighted']:.3f})")
    ax1.set_xscale("log"); ax1.set_ylim(0, 1.05)
    ax1.tick_params(axis="y", labelcolor=c1)
    ax1.grid(True, which="both", linestyle="--", alpha=0.35)

    ax2 = ax1.twinx(); c2 = "tab:red"
    ax2.set_ylabel("Adversary Accuracy", color=c2, fontsize=12)
    ax2.plot(sweep["epsilon"], sweep["adversary_accuracy"], color=c2, marker="x",
             linestyle="--", linewidth=2.5, markersize=8, label="Adversary")
    ax2.axhline(sweep["random_baseline"].iloc[0], color="grey", linestyle=":",
                linewidth=1.2,
                label=f"Random baseline ({sweep['random_baseline'].iloc[0]:.3f})")
    ax2.set_ylim(0, 1.05); ax2.tick_params(axis="y", labelcolor=c2)

    l1,lb1 = ax1.get_legend_handles_labels()
    l2,lb2 = ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, loc="center right", fontsize=9)

    plt.title(f"Privacy-Utility Tradeoff — {title}\n"
              f"Graph edge ε-DP → perturbed graph smoothing → PCA → cloud", fontsize=13)
    plt.tight_layout()
    plt.savefig("sweep_graph_edge_dp.png", dpi=300, bbox_inches="tight")
    plt.savefig("sweep_graph_edge_dp.pdf", bbox_inches="tight")
    plt.show()
    print("  Saved: sweep_graph_edge_dp.png")


def plot_flip_rate_vs_privacy(df):
    sweep = df[df["epsilon"] > 0].copy()
    fig, ax1 = plt.subplots(figsize=(10, 5))
    c1 = "tab:purple"
    ax1.set_xlabel("ε (log scale)", fontsize=12)
    ax1.set_ylabel("Edge Flip Rate", color=c1, fontsize=12)
    ax1.plot(sweep["epsilon"], sweep["flip_rate"], color=c1, marker="D",
             linewidth=2, markersize=6, label="Flip Rate")
    ax1.set_xscale("log"); ax1.set_ylim(0, 1.05)
    ax1.tick_params(axis="y", labelcolor=c1)
    ax1.grid(True, which="both", linestyle="--", alpha=0.35)

    ax2 = ax1.twinx(); c2 = "tab:red"
    ax2.set_ylabel("Privacy Gain", color=c2, fontsize=12)
    ax2.plot(sweep["epsilon"], sweep["privacy_gain"], color=c2, marker="x",
             linestyle="--", linewidth=2, markersize=7, label="Privacy Gain")
    ax2.axhline(0, color="black", linestyle="-", linewidth=0.8, alpha=0.5)
    ax2.tick_params(axis="y", labelcolor=c2)

    l1,lb1 = ax1.get_legend_handles_labels()
    l2,lb2 = ax2.get_legend_handles_labels()
    ax1.legend(l1+l2, lb1+lb2, fontsize=9)
    plt.title("Edge Flip Rate vs Privacy Gain\n"
              "Shows how graph perturbation intensity relates to privacy protection", fontsize=12)
    plt.tight_layout()
    plt.savefig("flip_rate_vs_privacy.png", dpi=200, bbox_inches="tight")
    plt.show()
    print("  Saved: flip_rate_vs_privacy.png")


def plot_confusion_matrices(results, class_names=None, label=""):
    n = len(results); ncols = min(3,n); nrows = int(np.ceil(n/ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 4.5*nrows))
    axes = np.array(axes).flatten()
    for idx, (name, m) in enumerate(results.items()):
        ConfusionMatrixDisplay(m["conf_matrix"], display_labels=class_names).plot(
            ax=axes[idx], colorbar=False, cmap="Blues")
        axes[idx].set_title(f"{name}\nAcc={m['accuracy']:.3f}  F1={m['f1_weighted']:.3f}", fontsize=10)
        axes[idx].tick_params(axis="x", rotation=45)
    for idx in range(len(results), len(axes)): axes[idx].set_visible(False)
    plt.suptitle(f"Confusion Matrices — {label}", fontsize=13, y=1.01)
    plt.tight_layout()
    fname = f"confusion_{label.replace(' ','_').replace('=','')}.png"
    plt.savefig(fname, dpi=200, bbox_inches="tight"); plt.show()


# ── Generate plots ────────────────────────────────────────────────────────
plot_graph_privacy_tradeoff(sweep_df)
plot_flip_rate_vs_privacy(sweep_df)

# Confusion matrix at epsilon_highlight
eps_h = CFG["epsilon_highlight"]
print(f"\nGenerating confusion matrices at ε={eps_h}...")
W_h, _   = perturb_graph_edges(W_clean, epsilon=eps_h)
L_h      = sparse_laplacian(W_h, normed=False).tocsr()
X_h      = smooth_features_sparse(X_all_sc, L_h, CFG["alpha"])
Z_h      = pca_ref.transform(X_h)
res_h    = run_classifiers(Z_h[:n_train], Z_h[n_train:],
                            y_train, y_test,
                            class_names=CFG["class_names"], verbose=False)
plot_confusion_matrices(res_h, CFG["class_names"], label=f"Graph ε={eps_h}")
print("\n✓ All plots saved.")